# R&D — Extraction texte depuis un AO en PDF

Objectif : valider l'approche en 2 phases avant industrialisation.

**Phase 1** : `pymupdf` extrait le texte embarqué (rapide, 0 token LLM)  
**Phase 2** : si le PDF est scanné (texte insuffisant), on rend les pages en images et on envoie à un LLM vision (Claude ou GPT-4o)

---
PDF de test : `../documents/test_ocr.pdf`

## 0. Installation des dépendances

In [ ]:
%pip install pymupdf python-dotenv anthropic openai --quiet

## 1. Chargement de la configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Charge le .env à la racine du projet
env_path = Path("..") / ".env"
load_dotenv(env_path)

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "")

print(f"Anthropic : {'OK' if ANTHROPIC_API_KEY else 'MANQUANT'}")
print(f"OpenAI    : {'OK' if OPENAI_API_KEY    else 'MANQUANT'}")

PDF_PATH = Path("../documents/test_ocr.pdf")
print(f"\nPDF       : {PDF_PATH} — {'trouvé ✓' if PDF_PATH.exists() else 'INTROUVABLE ✗'}")

## 2. Phase 1 — Extraction texte embarqué avec pymupdf

In [ ]:
import fitz  # pymupdf

# Seuil : si moins de N caractères en moyenne par page → PDF scanné
SCAN_THRESHOLD_CHARS_PER_PAGE = 50

def extract_text_pymupdf(pdf_path: Path) -> tuple[str, dict]:
    """
    Extrait le texte embarqué d'un PDF via pymupdf.
    Retourne (texte_complet, métadonnées).
    """
    doc = fitz.open(str(pdf_path))
    pages_text = []
    
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")  # mode "text" = texte brut sans mise en forme
        pages_text.append(text)
    
    full_text = "\n\n".join(pages_text)
    total_chars = len(full_text.strip())
    avg_chars   = total_chars / len(doc) if len(doc) > 0 else 0
    
    meta = {
        "pages":       len(doc),
        "total_chars": total_chars,
        "avg_chars_per_page": round(avg_chars, 1),
        "is_scanned":  avg_chars < SCAN_THRESHOLD_CHARS_PER_PAGE,
    }
    
    doc.close()
    return full_text, meta


text_phase1, meta = extract_text_pymupdf(PDF_PATH)

print(f"Pages              : {meta['pages']}")
print(f"Caractères total   : {meta['total_chars']:,}")
print(f"Caractères/page    : {meta['avg_chars_per_page']}")
print(f"Seuil scanné       : {SCAN_THRESHOLD_CHARS_PER_PAGE} car/page")
print(f"Détection          : {'SCANNÉ → fallback vision' if meta['is_scanned'] else 'PDF texte → extraction directe ✓'}")

In [ ]:
# Aperçu du texte extrait
print("=== TEXTE EXTRAIT (500 premiers caractères) ===")
print(text_phase1[:500])
print("\n...")
print("=== FIN (200 derniers caractères) ===")
print(text_phase1[-200:])

## 3. Phase 2 — Fallback vision LLM (si PDF scanné)

Si `is_scanned = True` : on rend chaque page en PNG (DPI 150) et on envoie au LLM vision.

On peut tester cette phase même sur un PDF texte pour comparer la qualité.

In [ ]:
import base64
from IPython.display import Image, display

def render_pages_as_images(pdf_path: Path, dpi: int = 150) -> list[bytes]:
    """
    Rend chaque page du PDF en PNG bytes.
    DPI 150 = bon compromis qualité/taille (environ 80-150 KB/page).
    """
    doc = fitz.open(str(pdf_path))
    images = []
    mat = fitz.Matrix(dpi / 72, dpi / 72)  # 72 DPI natif → facteur d'échelle
    
    for page in doc:
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        images.append(pix.tobytes("png"))
    
    doc.close()
    return images


# Rendu des pages
page_images = render_pages_as_images(PDF_PATH, dpi=150)

print(f"Pages rendues : {len(page_images)}")
for i, img in enumerate(page_images, start=1):
    kb = len(img) / 1024
    print(f"  Page {i} : {kb:.0f} KB")

# Aperçu visuel de la première page
print("\n--- Aperçu page 1 ---")
display(Image(data=page_images[0], width=600))

### 3a. Fallback via Claude (Anthropic vision)

In [ ]:
import anthropic

def extract_text_claude_vision(page_images: list[bytes], api_key: str) -> str:
    """
    Envoie les pages PNG à Claude claude-3-5-sonnet pour extraction de texte.
    Retourne le texte extrait complet.
    """
    client = anthropic.Anthropic(api_key=api_key)
    
    # Construction du contenu multimodal : une image par page
    content = []
    for i, img_bytes in enumerate(page_images, start=1):
        b64 = base64.standard_b64encode(img_bytes).decode()
        content.append({"type": "text", "text": f"--- Page {i} ---"})
        content.append({
            "type": "image",
            "source": {
                "type":       "base64",
                "media_type": "image/png",
                "data":       b64,
            },
        })
    
    content.append({
        "type": "text",
        "text": (
            "Extrais tout le texte de ces pages de document, en conservant la structure "
            "(titres, numéros de listes, tableaux). Ne résume pas, retranscris fidèlement."
        ),
    })
    
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=4096,
        messages=[{"role": "user", "content": content}],
    )
    
    return response.content[0].text


if not ANTHROPIC_API_KEY:
    print("ANTHROPIC_API_KEY manquante — skip")
else:
    print("Appel Claude vision...")
    text_claude = extract_text_claude_vision(page_images, ANTHROPIC_API_KEY)
    print(f"Texte extrait : {len(text_claude)} caractères\n")
    print(text_claude[:800])

### 3b. Fallback via GPT-4o (OpenAI vision)

In [ ]:
from openai import OpenAI

def extract_text_gpt4o_vision(page_images: list[bytes], api_key: str) -> str:
    """
    Envoie les pages PNG à GPT-4o pour extraction de texte.
    """
    client = OpenAI(api_key=api_key)
    
    content = []
    for i, img_bytes in enumerate(page_images, start=1):
        b64 = base64.standard_b64encode(img_bytes).decode()
        content.append({"type": "text", "text": f"--- Page {i} ---"})
        content.append({
            "type":      "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64}"},
        })
    
    content.append({
        "type": "text",
        "text": (
            "Extrais tout le texte de ces pages de document, en conservant la structure "
            "(titres, numéros de listes, tableaux). Ne résume pas, retranscris fidèlement."
        ),
    })
    
    response = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=4096,
        messages=[{"role": "user", "content": content}],
    )
    
    return response.choices[0].message.content


if not OPENAI_API_KEY:
    print("OPENAI_API_KEY manquante — skip")
else:
    print("Appel GPT-4o vision...")
    text_gpt4o = extract_text_gpt4o_vision(page_images, OPENAI_API_KEY)
    print(f"Texte extrait : {len(text_gpt4o)} caractères\n")
    print(text_gpt4o[:800])

## 4. Comparaison des résultats

In [ ]:
results = {}

results["pymupdf (phase 1)"] = text_phase1

try:
    results["claude vision"] = text_claude
except NameError:
    pass

try:
    results["gpt-4o vision"] = text_gpt4o
except NameError:
    pass

print("=" * 60)
print("BILAN COMPARATIF")
print("=" * 60)
for name, text in results.items():
    chars = len(text.strip())
    words = len(text.split())
    print(f"\n{name}")
    print(f"  Caractères : {chars:,}")
    print(f"  Mots       : {words:,}")

## 5. Fonction finale — logique d'arbitrage

C'est exactement le code qui sera industrialisé dans `app/services/pdf_extract_service.py`.

In [ ]:
SCAN_THRESHOLD = 50  # chars/page en dessous duquel on considère le PDF comme scanné
VISION_PROVIDER = "anthropic"  # "anthropic" ou "openai"

def extract_ao_text(pdf_bytes: bytes, anthropic_key: str = "", openai_key: str = "") -> dict:
    """
    Extrait le texte d'un AO PDF.
    
    Retourne :
      {
        'text':     str,            # texte extrait
        'method':   str,            # 'pymupdf' | 'claude_vision' | 'gpt4o_vision'
        'pages':    int,
        'is_scanned': bool,
      }
    """
    # Phase 1 : tentative d'extraction directe
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    pages_text = [page.get_text("text") for page in doc]
    n_pages    = len(doc)
    doc.close()
    
    full_text   = "\n\n".join(pages_text)
    avg_chars   = len(full_text.strip()) / n_pages if n_pages > 0 else 0
    is_scanned  = avg_chars < SCAN_THRESHOLD
    
    if not is_scanned:
        return {"text": full_text, "method": "pymupdf", "pages": n_pages, "is_scanned": False}
    
    # Phase 2 : rendu en images + vision LLM
    doc     = fitz.open(stream=pdf_bytes, filetype="pdf")
    mat     = fitz.Matrix(150 / 72, 150 / 72)
    images  = [page.get_pixmap(matrix=mat, colorspace=fitz.csRGB).tobytes("png") for page in doc]
    doc.close()
    
    if anthropic_key:
        text = extract_text_claude_vision(images, anthropic_key)
        return {"text": text, "method": "claude_vision", "pages": n_pages, "is_scanned": True}
    
    if openai_key:
        text = extract_text_gpt4o_vision(images, openai_key)
        return {"text": text, "method": "gpt4o_vision", "pages": n_pages, "is_scanned": True}
    
    raise RuntimeError("PDF scanné détecté mais aucune clé API vision disponible.")


# Test sur le fichier de référence
pdf_bytes = PDF_PATH.read_bytes()
result = extract_ao_text(pdf_bytes, anthropic_key=ANTHROPIC_API_KEY, openai_key=OPENAI_API_KEY)

print(f"Méthode   : {result['method']}")
print(f"Pages     : {result['pages']}")
print(f"Scanné    : {result['is_scanned']}")
print(f"Texte     : {len(result['text']):,} caractères")
print("\n--- Début du texte extrait ---")
print(result["text"][:600])

## 6. Conclusion

| Critère | pymupdf seul | + vision LLM fallback |
|---|---|---|
| PDF texte classique | ✅ Instantané, 0 token | ✅ Idem |
| PDF scanné | ❌ Texte vide | ✅ Fonctionne |
| Coût tokens | 0 | ~1000-3000 tokens/page |
| Dépendances | `pymupdf` | `pymupdf` uniquement |
| Installation Docker | `pip install pymupdf` | idem (pas de Tesseract) |

**Si OK → prochaine étape** : créer `app/services/pdf_extract_service.py` + route `POST /api/v1/pdf/extract` + bouton upload dans le frontend.